# Лаба 2: Сравнение регуляризаций (L1/L2/ElasticNet)
Логистическая регрессия на Titanic и линейная регрессия на California Housing.


In [1]:

import sys, os
import torch
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('..')
os.environ.setdefault('MPLCONFIGDIR', 'outputs/mpl_cache')
os.makedirs('outputs/mpl_cache', exist_ok=True)

from src.data.preprocessing import load_titanic_preprocessed, load_california_preprocessed, SimpleTorchLoader
from src.models.logistic_regression import LogisticRegression
from src.models.linear_regression import LinearRegression
from src.utils.visualization import plot_variant_curves

plt.style.use('seaborn-v0_8')
torch.manual_seed(42)
np.random.seed(42)


## Titanic: логистическая регрессия с регуляризацией


In [2]:

df, features, target = load_titanic_preprocessed()
loader = SimpleTorchLoader(df, target, features, standardize=True)
split = loader.split()
variants = [
    {'label': 'No reg', 'reg_type': None, 'reg_lambda': 0.0, 'elastic_alpha': 0.0},
    {'label': 'L2 λ=0.2', 'reg_type': 'l2', 'reg_lambda': 0.2, 'elastic_alpha': 0.0},
    {'label': 'L1 λ=0.2', 'reg_type': 'l1', 'reg_lambda': 0.2, 'elastic_alpha': 0.0},
    {'label': 'ElasticNet λ=0.2, ρ=0.5', 'reg_type': 'elasticnet', 'reg_lambda': 0.2, 'elastic_alpha': 0.5},
]

histories = []
for v in variants:
    model = LogisticRegression(
        learning_rate=5e-3,
        max_epochs=1000,
        reg_type=v['reg_type'],
        l1_lambda=v['reg_lambda'],
        l2_lambda=v['reg_lambda'],
        alpha=v['elastic_alpha'],
    )
    model.fit(split['X_train'], split['y_train'], split['X_val'], split['y_val'])
    histories.append((v['label'], model.history))

plot_variant_curves(histories, 'train_loss_data', 'Train BCE (data only)', '../outputs/lab2/logreg_train_bce.png', ylabel='BCE')
plot_variant_curves(histories, 'val_loss_data', 'Val BCE (data only)', '../outputs/lab2/logreg_val_bce.png', ylabel='BCE')
plot_variant_curves(histories, 'val_acc', 'Val Accuracy', '../outputs/lab2/logreg_val_acc.png', ylabel='Accuracy')
plot_variant_curves(histories, 'val_f1', 'Val F1', '../outputs/lab2/logreg_val_f1.png', ylabel='F1')


## California Housing: линейная регрессия с регуляризацией


In [3]:

df_c, features_c, target_c = load_california_preprocessed()
loader_c = SimpleTorchLoader(df_c, target_c, features_c, standardize=True)
split_c = loader_c.split()
variants = [
    {'label': 'No reg', 'reg_type': None, 'reg_lambda': 0.0, 'elastic_alpha': 0.0},
    {'label': 'L2 λ=1', 'reg_type': 'l2', 'reg_lambda': 1.0, 'elastic_alpha': 0.0},
    {'label': 'L1 λ=1', 'reg_type': 'l1', 'reg_lambda': 1.0, 'elastic_alpha': 0.0},
    {'label': 'ElasticNet λ=1, ρ=0.5', 'reg_type': 'elasticnet', 'reg_lambda': 1.0, 'elastic_alpha': 0.5},
]

histories_lin = []
for v in variants:
    model = LinearRegression(n_features=len(features_c), lr=5e-2, reg_type=v['reg_type'], reg_lambda=v['reg_lambda'], elastic_alpha=v['elastic_alpha'])
    model.fit(split_c['X_train'], split_c['y_train'], split_c['X_val'], split_c['y_val'], epochs=1000, print_every=200)
    histories_lin.append((v['label'], model.history))

plot_variant_curves(histories_lin, 'train_mse', 'Train MSE', '../outputs/lab2/linreg_train_mse.png', ylabel='MSE')
plot_variant_curves(histories_lin, 'val_mse', 'Val MSE', '../outputs/lab2/linreg_val_mse.png', ylabel='MSE')
plot_variant_curves(histories_lin, 'val_rmse', 'Val RMSE', '../outputs/lab2/linreg_val_rmse.png', ylabel='RMSE')
plot_variant_curves(histories_lin, 'val_mae', 'Val MAE', '../outputs/lab2/linreg_val_mae.png', ylabel='MAE')


[epoch   1] train_mse=56425533440.0 rmse=237540.6 mae=207481.2 | val_mse=44913905664.0 rmse=211929.0 mae=184386.0
[epoch 200] train_mse=4914208768.0 rmse=70101.4 mae=51282.7 | val_mse=4340385280.0 rmse=65881.6 mae=49435.4
[epoch 400] train_mse=4875492864.0 rmse=69824.7 mae=50891.3 | val_mse=4294938880.0 rmse=65535.8 mae=49113.8
[epoch 600] train_mse=4860867584.0 rmse=69719.9 mae=50752.1 | val_mse=4279925504.0 rmse=65421.1 mae=48985.1
[epoch 800] train_mse=4852813824.0 rmse=69662.1 mae=50682.2 | val_mse=4271607040.0 rmse=65357.5 mae=48909.1
[epoch 1000] train_mse=4847845888.0 rmse=69626.5 mae=50638.4 | val_mse=4266326016.0 rmse=65317.1 mae=48859.0
[epoch   1] train_mse=56425533440.0 rmse=237540.6 mae=207481.2 | val_mse=44913905664.0 rmse=211929.0 mae=184386.0
[epoch 200] train_mse=4914617856.0 rmse=70101.7 mae=51283.2 | val_mse=4340831744.0 rmse=65882.2 mae=49435.9
[epoch 400] train_mse=4875976192.0 rmse=69825.0 mae=50892.0 | val_mse=4295470080.0 rmse=65536.5 mae=49114.5
[epoch 600] tra